****Problem Set****

In [215]:
import pandas as pd
import numpy as np
import scipy.stats as stats

In [216]:
## load in data
ames = pd.read_csv("../data/ames_prices.csv")
meta = pd.read_csv("../data/metabric.csv")
nhanes = pd.read_csv("../data/nhanes_data_17_18.csv")

# recode Central.Air as binary variable
ames['central_air_binary'] = ames['Central.Air'].map({'Y': 1, 'N': 0})
# recode Overall Survival Status as a binary variable
meta['deceased'] = meta['Overall Survival Status'].map({'1:DECEASED': 1, '0:LIVING': 0})


/var/folders/9z/540dspmj1j31q6xfjkl6q6gm0000gn/T/ipykernel_28227/1399288662.py:4: DtypeWarning: Columns (142: ServedActiveDutyInUsArmedForces) have mixed types. Specify dtype option on import or set low_memory=False.
  nhanes = pd.read_csv("../data/nhanes_data_17_18.csv")


In [217]:
# Set up bootstrap variables 
R = 2_000
seed = 5030
rng = np.random.default_rng(seed)
print(f'Seed: {seed}\nBootstrap Iterations (R): {R}')


Seed: 5030
Bootstrap Iterations (R): 2000


**Q7**

In [218]:
## Parameters

# population proportion
p = .9

# critical values
z_crits = [1.645, 1.96, 2.576]

# confidence levels
conf_lev = [90, 95, 99]

# significance levels
alphas = [.10, .05, .01]

In [219]:
# sample proportion (one sample)
p_hat = sum(ames['central_air_binary'])/len(ames)
print(f'sample proportion: {p_hat},\npopulation proportion: {p}')

sample proportion: 0.9331058020477816,
population proportion: 0.9


In [220]:
# bootstrap sample proportions
prop_boot = []
for i in range(R):
    df_r = ames['central_air_binary'].sample(frac = 1.0, 
                       replace = True, 
                       random_state=rng)
    theta = df_r.mean()
    prop_boot.append(theta)


In [221]:
# standard error
se = np.std(prop_boot, ddof=1)
print(f'standard error: {se}')

standard error: 0.004618435554234278


In [222]:
# confidence intervals and test
for i in range(len(conf_lev)):
    low = p_hat - z_crits[i]*se
    high = p_hat + z_crits[i]*se
    conf = conf_lev[i]
    print(f'CI at {conf}%: ({low},{high}) \n contains null {p}: {p > low and p < high}')

CI at 90%: (0.9255084755610662,0.940703128534497) 
 contains null 0.9: False
CI at 95%: (0.9240536683614824,0.9421579357340808) 
 contains null 0.9: False
CI at 99%: (0.921208712060074,0.9450028920354892) 
 contains null 0.9: False


In [223]:
# z score
z = (p_hat - p)/se 

# z score vs critical value test
for c in z_crits:
    print(f'z-score {z} more extreme than {c}: {abs(z) > c}')

z-score 7.16818534307131 more extreme than 1.645: True
z-score 7.16818534307131 more extreme than 1.96: True
z-score 7.16818534307131 more extreme than 2.576: True


In [224]:
# p-value
p_val = 2*(1-stats.t.cdf(abs(z), R - 1))

# p-value test
for a in alphas:
    print(f'The p-value {p_val} is less than {a}: {p_val < a}')

The p-value 1.0647038806155251e-12 is less than 0.1: True
The p-value 1.0647038806155251e-12 is less than 0.05: True
The p-value 1.0647038806155251e-12 is less than 0.01: True


### **Confirm and Interpret**

In all of the tests for all confidence and significance levels, the resulting conclusion is that there is significant evidence to reject the null hypothesis. Since the test is significant at all levels, we will interpret with the strictest significance level, 0.01. We are 99% confident that the true proportion of homes with central air is contained within the interval (0.9212, 0.9450) while the null is not, and there is significant evidence to reject the null hypothesis that the true proportion of homes with central air is 0.90 at the 0.01 significance level because the resulting p-value is 0.0000000000011 and much smaller than 0.01. Our estimated sample proportion is 0.933 which is not equal to the null proportion 0.90, so it is one of the first indicators that the null might not be 0.90, but is not enough evidence to conclude anything so we perform hypothesis testing. 

When we say we are 99% confident an interval contains the true value, we mean that 99% of generated intervals will contain the true value given the parameters and statistics of the study such as the standard error, sample size, etc. We can interpret the p-value to mean that there is less than a 0.001 probability that we achieve such an extreme statistic (sample proportion) given the null hypothesis that the proportion is 0.90 is true. 

**Q8**

In [225]:
# pearson correlation
r_pop = 0
r_pear = np.arcsinh(ames['area']).corr(np.arcsinh(ames['price']))
print(f'null correlation: {r_pop}\nsample correlation: {r_pear}')

null correlation: 0
sample correlation: 0.7231661530091354


In [226]:
# bootstrap paired rows
corr_boot = []
for i in range(R):
    df_r = ames[['area', 'price']].sample(frac = 1.0, 
                       replace = True, 
                       random_state=rng)
    theta = np.arcsinh(df_r['area']).corr(np.arcsinh(df_r['price']))
    corr_boot.append(theta)

In [227]:
# standard error
se = np.std(corr_boot, ddof=1)
print(f'standard error: {se}')

standard error: 0.010092339234403967


In [228]:
# test confidence intervals
for i in range(len(conf_lev)):
    low = r_pear - z_crits[i]*se
    high = r_pear + z_crits[i]*se
    conf = conf_lev[i]
    print(f'CI at {conf}%: ({low},{high}) \n contains null {r_pop}: {r_pop > low and r_pop < high}')

CI at 90%: (0.7065642549685409,0.73976805104973) 
 contains null 0: False
CI at 95%: (0.7033851681097036,0.7429471379085673) 
 contains null 0: False
CI at 99%: (0.6971682871413108,0.74916401887696) 
 contains null 0: False


In [229]:
# test z score
z = (r_pear - r_pop)/se 

for c in z_crits:
    print(f'z-score {z} more extreme than {c}: {abs(z) > c}')

z-score 71.65495889633996 more extreme than 1.645: True
z-score 71.65495889633996 more extreme than 1.96: True
z-score 71.65495889633996 more extreme than 2.576: True


In [230]:
# test p-value
p_val = 2*(1-stats.t.cdf(abs(z), R - 1))

for a in alphas:
    print(f'The p-value {p_val} is less than {a}: {p_val < a}')

The p-value 0.0 is less than 0.1: True
The p-value 0.0 is less than 0.05: True
The p-value 0.0 is less than 0.01: True


### **Confirm and Interpret**

All the tests show significant evidence that the null hypothesis should be rejected so we will interpret at the 99% confidence level and 0.01 significance level. There is significant evidence that the Pearson correlation between arcsinh(area) and arcsinh(price) (the transformed area and price of homes) is not 0, so we reject our null hypothesis at the 0.01 level because our p-value is less than 0.001. We are 99% confident that the true correlation between price and area of homes is contained in the interval (0.697, 0.749), which does not include 0. Our sample estimate of the correlation is 0.723, which is much higher than 0 (the null) so our results are not surprising. Practically, it means that there is evidence against the null hpothesis that there is no relationship or correlation between home area and home price.

If area and price were resampled separately, then it would be impossible to study the link between them and measure how the price and area for the same home interact. Instead, the price of one home would be compared to the area of another home, which does not give us useful information on the relationship between the variables. The true correlation would likely be near 0 if resampled independently. 

**Q9**

In [231]:
# sample and population mean
mu = 120
x_bar = meta['Overall Survival (Months)'].mean()
print(f'population mean: {mu}\nsample mean: {x_bar}')


population mean: 120
sample mean: 129.09429138744105


In [232]:
# bootstrap mean
mean_boot = []
for i in range(R):
    df_r = meta["Overall Survival (Months)"].sample(frac = 1.0, 
                       replace = True, 
                       random_state=rng)
    theta = df_r.mean()
    mean_boot.append(theta)

In [233]:
# standard error
se = np.std(mean_boot, ddof=1)
print(f'standard error: {se}')

standard error: 2.1303931244027927


In [234]:
# test confidence intervals
for i in range(len(conf_lev)):
    low = x_bar - z_crits[i]*se
    high = x_bar + z_crits[i]*se
    conf = conf_lev[i]
    print(f'CI at {conf}%: ({low},{high}) \n contains null {mu}: {mu > low and mu < high}')

CI at 90%: (125.58979469779845,132.59878807708364) 
 contains null 120: False
CI at 95%: (124.91872086361157,133.2698619112705) 
 contains null 120: False
CI at 99%: (123.60639869897945,134.58218407590263) 
 contains null 120: False


In [235]:
# test z score
z = (x_bar - mu)/se 

for c in z_crits:
    print(f'z-score {z} more extreme than {c}: {abs(z) > c}')

z-score 4.268832490712447 more extreme than 1.645: True
z-score 4.268832490712447 more extreme than 1.96: True
z-score 4.268832490712447 more extreme than 2.576: True


In [236]:
# test p-value
p_val = 2*(1-stats.t.cdf(abs(z), R - 1))

for a in alphas:
    print(f'The p-value {p_val} is less than {a}: {p_val < a}')

The p-value 2.0569341772080918e-05 is less than 0.1: True
The p-value 2.0569341772080918e-05 is less than 0.05: True
The p-value 2.0569341772080918e-05 is less than 0.01: True


### **Confirm and Interpret**

All of the tests indicate that we should reject the null hypothesis. We are 99% confident that the true amount of months to death or last doctor follow-up for cancer patients is contained within the interval (123.606, 134.582) which does not contain the null amount of time (120 months). In years, the interval is (10.30, 11.22), and the null amount of time is 10 years. At the 0.01 significance level with a p-value of 0.00002, we reject the null hypothesis that the average number of months to time of death or last follow up for patients is 120 months (or 10 years) meaning that the true mean is either above or below 120 months. 

Since the study includes patients who have both died and are still living, the interpretation of survival length is not very clear. That is why we must specify "time to death or last doctor follow up" in our interpretation and conclusion, rather than explain the amount of time as an expected lifetime. If the study only included data on patients who died, then the interpretation could be related to expected lifetime.  

**Q10**

In [237]:
# sample and population mean
p_hata = meta[meta['Chemotherapy'] == 'YES']['deceased'].mean()
p_hatb = meta[meta['Chemotherapy'] == 'NO']['deceased'].mean()
p_hat_diff = p_hata - p_hatb
pop_diff = 0

print(f'''sample proportion chemotherapy: {p_hata}\n
sample proportion no chemotherapy: {p_hatb}\n
sample proportion difference: {abs(p_hat_diff)}\n
null difference: {pop_diff}''')


sample proportion chemotherapy: 0.5209790209790209

sample proportion no chemotherapy: 0.5856196783349101

sample proportion difference: 0.0646406573558892

null difference: 0


In [238]:
# bootstrap mean
diff_boot = []
for i in range(R):
    df_a = meta[meta['Chemotherapy'] == 'YES'].sample(frac = 1.0, 
                       replace = True, 
                       random_state=rng)
    df_b = meta[meta['Chemotherapy'] == 'NO'].sample(frac = 1.0, 
                       replace = True, 
                       random_state=rng)
    diff_a = df_a['deceased'].mean()
    diff_b = df_b['deceased'].mean()
    theta = diff_a - diff_b
    diff_boot.append(theta)


In [239]:
# standard error
se = np.std(diff_boot, ddof=1)
print(f'standard error: {se}')


standard error: 0.03258789576735016


In [240]:
# test confidence intervals
for i in range(len(conf_lev)):
    low = p_hat_diff - z_crits[i]*se
    high = p_hat_diff + z_crits[i]*se
    conf = conf_lev[i]
    print(f'CI at {conf}%: ({low},{high}) \n contains 0: {0 > low and 0 < high}')


CI at 90%: (-0.11824774589318021,-0.011033568818598183) 
 contains 0: False
CI at 95%: (-0.12851293305989553,-0.0007683816518828795) 
 contains 0: False
CI at 99%: (-0.1485870768525832,0.019305762140804816) 
 contains 0: True


In [241]:
# test z score
z = (p_hat_diff - pop_diff)/se 

for c in z_crits:
    print(f'z-score {z} more extreme than {c}: {abs(z) > c}')


z-score -1.9835787440026345 more extreme than 1.645: True
z-score -1.9835787440026345 more extreme than 1.96: True
z-score -1.9835787440026345 more extreme than 2.576: False


In [242]:
# test p-value
p_val = 2*(1-stats.t.cdf(abs(z), R - 1))

for a in alphas:
    print(f'The p-value {p_val} is less than {a}: {p_val < a}')

The p-value 0.0474394292576128 is less than 0.1: True
The p-value 0.0474394292576128 is less than 0.05: True
The p-value 0.0474394292576128 is less than 0.01: False


In [243]:
meta.head()

,Age at Diagnosis,Type of Breast Surgery,Cancer Type,Chemotherapy,Hormone Therapy,Lymph nodes examined positive,Mutation Count,Nottingham prognostic index,Overall Survival (Months),Overall Survival Status,Radio Therapy,TMB (nonsynonymous),Tumor Size,Tumor Stage,deceased
0,43.19,BREAST CONSERVING,Breast Cancer,NO,YES,0.0,2.0,4.020,84.633333,0:LIVING,YES,2.615035,10.0,1.0,0
1,48.87,MASTECTOMY,Breast Cancer,YES,YES,1.0,2.0,4.030,163.700000,1:DECEASED,NO,2.615035,15.0,2.0,1
2,47.68,MASTECTOMY,Breast Cancer,YES,YES,3.0,1.0,4.050,164.933333,0:LIVING,YES,1.307518,25.0,2.0,0
3,76.97,MASTECTOMY,Breast Cancer,YES,YES,8.0,2.0,6.080,41.366667,1:DECEASED,YES,2.615035,40.0,2.0,1
4,78.77,MASTECTOMY,Breast Cancer,NO,YES,0.0,4.0,4.062,7.800000,1:DECEASED,YES,5.230071,31.0,4.0,1


In [244]:
meta['Chemotherapy'].value_counts()

Chemotherapy
NO     1057
YES     286
Name: count, dtype: int64

### **Confirm and Interpret**

Only the tests at the 99% confidence level and the 0.01 alpha level failed to reject the null hypothesis. 

We are 95% confident that the true difference in death proportions between patients who underwent chemotherapy versus those who did not is contained within the interval (-0.12851293305989553,-0.0007683816518828795), where the null proportion 0 is not included. We reject the null hypothesis that there is no difference in proportion of deaths between those who went through chemotherapy and those who did not at the 0.05 significance level with a p-value of 0.047. 

We fail to reject the null hypothesis at a significance level of 0.01 because it is less than our p-value which means our sample proportion is not extreme enough to be considered significant evidence against the null. We are 99% confident that the true proportion difference is within the interval (-0.149, 0.019), which contains the null proportion 0 which is why the evidence is not significant. 

One possible confounder in the data set are the fact that not all of the patients have died yet, some are still alive so we cannot attribute their survival to chemotherapy or the lack of it because each patient is operating on a different time line. Another confounder is there are different numbers of patients who received chemotherapy compared to those who didn't. Many more did not receive it (1057) than those who did (286), which will introduce higher variance and make it difficult to make the comparison between the two groups meaningful. 


**Q11**

In [245]:
# filter data
sleep = nhanes[nhanes['AgeInYearsAtScreening'] >= 18]['SleepHoursWeekdaysOrWorkdays']

In [246]:
# sample and population mean
x_bar = sleep.mean()
mu = 8

print(f'sample mean: {x_bar}\nnull mean: {mu}')


sample mean: 7.62124248496994
null mean: 8


In [247]:
# bootstrap mean
mean_boot = []
for i in range(R):
    df_a = sleep.sample(frac = 1.0, 
                       replace = True, 
                       random_state=rng)
    theta = df_a.mean()
    mean_boot.append(theta)


In [248]:
# standard error
se = np.std(mean_boot, ddof=1)
print(f'standard error: {se}')


standard error: 0.022847554884688533


In [249]:
# test confidence intervals
for i in range(len(conf_lev)):
    low = x_bar - z_crits[i]*se
    high = x_bar + z_crits[i]*se
    conf = conf_lev[i]
    print(f'CI at {conf}%: ({low},{high}) \n contains null {mu}: {mu > low and mu < high}')


CI at 90%: (7.583658257184628,7.6588267127552525) 
 contains null 8: False
CI at 95%: (7.576461277395951,7.66602369254393) 
 contains null 8: False
CI at 99%: (7.562387183586982,7.680097786352898) 
 contains null 8: False


In [250]:
# test z score
z = (x_bar - mu)/se 

for c in z_crits:
    print(f'z-score {z} more extreme than {c}: {abs(z) > c}')


z-score -16.577595149312323 more extreme than 1.645: True
z-score -16.577595149312323 more extreme than 1.96: True
z-score -16.577595149312323 more extreme than 2.576: True


In [251]:
# test p-value
p_val = 2*(1-stats.t.cdf(abs(z), R - 1))

for a in alphas:
    print(f'The p-value {p_val} is less than {a}: {p_val < a}')

The p-value 0.0 is less than 0.1: True
The p-value 0.0 is less than 0.05: True
The p-value 0.0 is less than 0.01: True


### **Confirm and Interpret**

All of the tests showed significance evidence for rejecting the null hypothesis. 

We are 99% confident that the true mean hours of sleep is contained within the interval (). This means 99% of generated confidence intervals will contain the true value through resampling with replacement. The null mean, 8, is not included in the interval which is our evidence that it is not the true value and so we reject the null hypothesis. At a significance level of 0.01, we again reject the null hypothesis with a p-value of less than 0.001, which means that we have a less tgab 0.001 probability of seeing an estimated parameter (statistic) as extreme as ours given the null hypothesis is true. 

Statistical and practical significance do not always tell the same story. In this case, the confidence intervals for all confidence levels tend to be between 7.5 and 7.7, which is not that big of a range and in practice can be rounded to 8 hours anyway. Statistically, 7.62 hours of sleep is different from 8 hours, but practically, you will probably feel pretty similar after waking up with either amount of sleep. 